In [1]:
!pip install transformers
!pip install datasets
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_syst

In [2]:
import pandas as pd
from transformers import pipeline
from datasets import Dataset

In [3]:
pipe = pipeline(
    "token-classification",
    model="edwardjross/xlm-roberta-base-finetuned-recipe-all",
    batch_size=16
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
df = pd.read_csv("/content/drive/MyDrive/unique_ingredients.tsv", sep="\t")

In [5]:
df.head()

,Ingredient,IDs
0,low sodium vegetable,"0, 11434, 14768, 96363, 105999, 194867, 255566..."
1,chicken stock,"0, 11, 94, 101, 130, 155, 345, 678, 852, 876, ..."
2,dried brown lentils,"0, 17104, 836751, 838386, 841486, 871267, 8744..."
3,dried French green lentils,"0, 2684, 3173, 3530, 16691"
4,"celery, chopped","0, 1149, 2427, 3530, 5081, 5340, 5891, 8907, 9..."


In [6]:
null_rows_df = df[df["Ingredient"].isna()]
null_rows_df

,Ingredient,IDs
372790,NaN,544237


In [7]:
print(df.info())
print(df["Ingredient"].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505063 entries, 0 to 505062
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Ingredient  505062 non-null  object
 1   IDs         505063 non-null  object
dtypes: object(2)
memory usage: 7.7+ MB
None
1


In [8]:
df = df.dropna(subset=["Ingredient"])
df["Ingredient"] = df["Ingredient"].astype(str).str.strip()
df = df[df["Ingredient"] != ""]

<ipython-input-8-cdd59af34696>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Ingredient"] = df["Ingredient"].astype(str).str.strip()


In [9]:
ds = Dataset.from_pandas(df)

In [10]:
def infer_batch(batch):
    # batch["text"] -> ['cümle1', 'cümle2', ...]
    predictions = pipe(batch["Ingredient"])  # toplu işlem
    return {"prediction": predictions}

In [11]:
ds_result = ds.map(
    infer_batch,
    batched=True,
    batch_size=16
)

Map:   0%|          | 0/505062 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [12]:
df_result = ds_result.to_pandas()
df_result.to_csv("/content/drive/MyDrive/output.tsv", sep="\t", index=False)